## Summary Step 2: Engineering and Transforming Features

In this step, I created new columns to make the data better for analysis.

#### What I did: 

- I created "Car Age" by subtracting the year from 2026 to see how old the car is.

- I calculated "Mileage per Year" to see how much the car was driven annually.

- I converted "Condition" words into numbers from 0 to 5 and encoded Fuel and Transmission.

#### I adjusted the data:

- I applied a Log Transformation to the price to make the charts look better.

- I used "StandardScaler" to make Odometer and Age have the same scale.

#### Final result:

- A dataset with new smart features ready for visualization.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Load Cleaned Data
df = pd.read_csv("data/cleaned/cars_cleaned.csv")

# Show data info before features
print("--- Data Info Before Features ---")
before_rows = df.shape[0]
before_cols = df.shape[1]
print(f"Total rows: {before_rows}")
print(f"Total columns: {before_cols}")


# Create car age and mileage per year
now_year = 2026
df["car_age"] = now_year - df["year"]
df["mileage_per_year"] = df["odometer"] / (df["car_age"] + 1)

# Convert condition to score numbers
condition_map = {
    'new': 5,
    'like new': 4,
    'excellent': 3,
    'good': 2,
    'fair': 1,
    'salvage': 0
}
df["condition_score"] = df["condition"].map(condition_map)

# Create interaction feature (condition * mileage)
df['condition_mileage_int'] = df['condition_score'] * df['odometer_scaled']

# Check correlation and remove redundant columns (r > 0.95)
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
df = df.drop(columns=to_drop)

# Convert fuel and transmission to 0 and 1
df = pd.get_dummies(df, columns=['fuel', 'transmission'], dtype=int)

# Apply log transformation to price
df['log_price'] = np.log1p(df['price'])

# Scale numerical columns
scaler = StandardScaler()
df[['odometer_scaled', 'car_age_scaled']] = scaler.fit_transform(df[['odometer', 'car_age']])

# Remove original condition column
df = df.drop(columns=['condition'])

# Show data info after features
print("\n--- Data Info After Features ---")
after_rows = df.shape[0]
after_cols = df.shape[1]
print(f"Total rows: {after_rows}")
print(f"Total columns: {after_cols}")

# Check data
print("\n--- Checks ---")

year_check = ('year' in df.columns) # Check if year column exists
print(f"Year column exists: {year_check}")
print(f"Total columns now: {after_cols}") # Check total columns count


# Save features data
df.to_csv("data/cleaned/cars_features.csv", index=False)
print("\nDone! Saved to cars_features.csv")

--- Data Info Before Features ---
Total rows: 128752
Total columns: 8

--- Data Info After Features ---
Total rows: 128752
Total columns: 19

--- Checks ---
Year column exists: True
Total columns now: 19

Done! Saved to cars_features.csv
